# 05 X打分
合并数据的冻结规则匹配复现。X是促销规则菜单中已记录类别的丰富度，共11类；不代表评论者已实际接触或使用规则。

建议Python 3.12，首次在本目录安装requirements.txt。所有路径相对本项目。new采集目录不会被读取。

本版使用2026-10-14早晨已定稿的23,913条合并池（原22,831＋新增1,082），不是此刻重新采集。旧版另存于并列目录，本项目不读写旧版。新采集输出new目录不会自动混入本版分析。

### 数据与来源边界
原始来源目录只覆盖本次23,913条：B站12,519、抖音10,910、小红书484、微博0。H的20,386条是历史整理源记录，不冒充原网页DOM；旧A的2,429条对应原文件SHA一致，另16条小红书评论数组一致但采集元数据有既有规范化差异。旧抖音3条原正文含“作者赞过”UI徽标，既有分析文本已去除，原文和变换均保留。

新增1,082条已逐条接上保存的捕获对象：991条原bundle SHA一致，91条为原捕获对象恢复，不能声称原文件字节也恢复。326条需要原生图像alt并入DOM正文，另756条不需alt，拼接后与冻结分析文本一致。另有1条连接原语义开发family而未入模，不把它混进本池。

23,384条有历史保存的规范来源链接，529条旧抖音记录未保存链接；本次未重新访问这些链接。日期保留来源显示精度和时区状态，不保证都是绝对日期。可核验原生评论ID共484个，均为小红书；其余本地row_id不冒充原生ID。

X规则材料共111个来源，47个来源有49份已存正文，64个来源没有完整正文，保留可得链接、描述和短引文状态。全池3,562条有冻结合格X（旧2,623＋新939），20,351条缺失；以下代码从规则事实重新计算数值，不为缺失行推造匹配。

In [1]:
from pathlib import Path
import sys
# 从python目录或项目根启动均可；没有固定机器路径
PYTHON_DIR = Path.cwd() if (Path.cwd() / 'analysis_helpers.py').exists() else Path.cwd() / 'python'
if not (PYTHON_DIR / 'analysis_helpers.py').exists():
    raise FileNotFoundError('请从lasso-新数据或lasso-新数据/python目录打开此Notebook')
sys.path.insert(0, str(PYTHON_DIR.resolve()))
import analysis_helpers as h
if h.ROOT.resolve() != PYTHON_DIR.resolve().parent:
    raise RuntimeError("当前内核已加载另一个项目版本；请重启内核后运行本Notebook")
print(h.runtime_check())

{'python': '3.12.14', 'numpy': '2.3.5', 'scipy': '1.17.0', 'scikit-learn': '1.8.0'}


## 1 读取已核对的规则材料
规则文本→语义事实→类别、来源等级与D1–D4材料覆盖，是历史人工核对步骤。程序从保留的实际规则CSV计算类别状态、材料门槛和X，不把冻结X数值当作计算输入。未知不等于现实不存在；冻结缺失匹配保持缺失。

In [2]:
x_rows, environment = h.compute_x()
print('全池:', len(x_rows), '有X:', int(x_rows.X.notna().sum()), '实际ENV:', x_rows.loc[x_rows.X.notna(), 'ENV'].nunique())
display(environment.head(8))

全池: 23913 有X: 3562 实际ENV: 27


,ENV,trading_platform,event,event_year,X_ENV,observed_category_count,unknown_category_count,document_gate_pass,source_admission_pass,coverage_rate,...,C2,C3,C4,C5,R1,R2,R3,R4,R5,R6
0,京东|双11|2022,京东,双11,2022,1.000000,11,0,True,True,1.000,...,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes
1,京东|双11|2023,京东,双11,2023,0.909091,10,1,True,True,1.000,...,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,unknown,observed_yes,observed_yes
2,淘宝|618|2021,淘宝,618,2021,0.727273,8,3,True,True,0.875,...,observed_yes,observed_yes,unknown,unknown,observed_yes,observed_yes,observed_yes,unknown,observed_yes,observed_yes
3,淘宝|双11|2021,淘宝,双11,2021,0.727273,8,3,True,True,0.750,...,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,unknown,unknown,unknown,observed_yes
4,京东|618|2023,京东,618,2023,1.000000,11,0,True,True,1.000,...,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes
5,淘宝|618|2023,淘宝,618,2023,0.636364,7,4,True,True,0.750,...,observed_yes,observed_yes,unknown,unknown,observed_yes,observed_yes,unknown,observed_yes,observed_yes,unknown
6,淘宝|618|2025,淘宝,618,2025,0.909091,10,1,True,True,0.750,...,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,unknown
7,淘宝|双11|2025,淘宝,双11,2025,0.727273,8,3,True,True,0.750,...,observed_yes,observed_yes,observed_yes,observed_yes,unknown,observed_yes,observed_yes,observed_yes,unknown,unknown


## 2 查看匹配结果
先按历史已核对的交易平台×活动×年份匹配ENV，再使用规则计算结果。信息来源平台（B站等）不等于交易平台（京东等）。只恢复冻结准入记录，不自动为冻结缺失记录补值。

In [3]:
display(x_rows.loc[x_rows.X.notna(), ['row_id','ENV','X']].head(10))
print(x_rows.groupby('stream').X.count().to_dict())

,row_id,ENV,X
0,BILI:BV11ZGX64EDa:checkpoint089_capture_1,抖音|618|2026,0.636364
1,BILI:BV11ZGX64EDa:checkpoint089_capture_10,抖音|618|2026,0.636364
2,BILI:BV11ZGX64EDa:checkpoint089_capture_11,抖音|618|2026,0.636364
3,BILI:BV11ZGX64EDa:checkpoint089_capture_12,抖音|618|2026,0.636364
5,BILI:BV11ZGX64EDa:checkpoint089_capture_14,抖音|618|2026,0.636364
6,BILI:BV11ZGX64EDa:checkpoint089_capture_15,抖音|618|2026,0.636364
7,BILI:BV11ZGX64EDa:checkpoint089_capture_16,抖音|618|2026,0.636364
8,BILI:BV11ZGX64EDa:checkpoint089_capture_17,抖音|618|2026,0.636364
9,BILI:BV11ZGX64EDa:checkpoint089_capture_18,抖音|618|2026,0.636364
10,BILI:BV11ZGX64EDa:checkpoint089_capture_19,抖音|618|2026,0.636364


{'A': 2858, 'H': 704}


## 3 可选导出
默认只校验和显示。开启后结果写回02_X匹配数据，不覆盖冻结匹配元数据。

In [4]:
WRITE_OUTPUT = False
if WRITE_OUTPUT:
    output = h.DATA / '02_X匹配数据' / 'X重新计算.csv'
    x_rows.to_csv(output, index=False, encoding='utf-8-sig')
    print(output.relative_to(h.ROOT))